# Elecciones presidenciales y votos nulos/blancos en Chile (1989–2025)

**Script de carga y exploración inicial de la base de datos limpia**

Este notebook carga la base de datos en formato CSV, visualiza sus primeras filas y realiza una exploración básica de sus variables. La base registra los resultados de las elecciones presidenciales chilenas desde el retorno a la democracia, con foco en la evolución del voto nulo y blanco.

---

## 0. Instalación e importación de librerías

Se utilizan las siguientes librerías:
- **pandas**: carga y manipulación del DataFrame
- **matplotlib** y **seaborn**: visualizaciones estáticas

En Google Colab estas librerías están preinstaladas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración visual general
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Librerías cargadas correctamente.')

---
## 1. Carga del CSV

El archivo `elecciones_presidenciales.csv` debe estar en el mismo directorio que este notebook, o en el drive montado si se trabaja en Google Colab.

> **Instrucción para Google Colab:** Subir el archivo CSV usando el panel lateral (ícono de carpeta → subir archivo), o montando Google Drive con `drive.mount('/content/drive')` y ajustando la ruta.

In [ ]:
# --- Si usas Google Colab y quieres subir el archivo manualmente, descomenta esto:
# from google.colab import files
# uploaded = files.upload()  # Abrirá un diálogo para seleccionar el archivo

# Ruta del archivo (ajustar si se usa Drive)
RUTA_CSV = 'elecciones_presidenciales.csv'

# Carga en DataFrame
df = pd.read_csv(RUTA_CSV, encoding='utf-8-sig')

print(f'Base cargada correctamente: {df.shape[0]} filas × {df.shape[1]} columnas.')

---
## 2. Primeras filas de la base

Visualización de las primeras filas del DataFrame para confirmar que la carga fue correcta y revisar la estructura general de los datos.

In [ ]:
df.head()

In [ ]:
# Ver todas las filas (la base es pequeña, 9 registros)
df

---
## 3. Exploración básica de la estructura

Revisión de tipos de datos, valores nulos y estadísticas descriptivas.

In [ ]:
# Tipos de datos y valores no nulos por columna
df.info()

In [ ]:
# Estadísticas descriptivas de las variables numéricas
df[['Votos totales', 'votos nulos', 'Votos blancos',
    'Suma votos nulos y blancos', '% votos nulos y blancos']].describe().round(4)

In [ ]:
# Formatear el porcentaje como % legible para análisis
df['% nulos y blancos (legible)'] = (df['% votos nulos y blancos'] * 100).round(2).astype(str) + '%'

# Vista resumida de las variables clave
df[['Año Elecciones', 'candidato 1', 'candidato 2',
    'Régimen del voto', 'Votos totales', '% nulos y blancos (legible)']]

---
## 4. Visualización: Evolución del voto nulo y blanco (1989–2025)

Gráfico de línea con el porcentaje de votos nulos y blancos por elección, destacando los cambios en el régimen de votación.

In [ ]:
# Etiquetas simplificadas para el eje X
etiquetas = [
    '1989', '1993', '1999',
    '2005\n(2ª vuelta)', '2009\n(2ª vuelta)',
    '2013\n(2ª vuelta)', '2017\n(2ª vuelta)',
    '2021\n(2ª vuelta)', '2025\n(2ª vuelta)'
]

pct = df['% votos nulos y blancos'] * 100
colores_regimen = [
    '#4C72B0', '#4C72B0', '#4C72B0',   # Inscripción voluntaria, voto obligatorio
    '#4C72B0', '#4C72B0',
    '#DD8452', '#DD8452', '#DD8452',   # Voluntario
    '#C44E52'                           # Obligatorio universal
]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(len(pct)), pct, color='#333333', linewidth=1.8, zorder=2)
ax.scatter(range(len(pct)), pct, color=colores_regimen, s=90, zorder=3)

for i, (x, y) in enumerate(zip(range(len(pct)), pct)):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=8.5, color='#333333')

ax.set_xticks(range(len(etiquetas)))
ax.set_xticklabels(etiquetas, fontsize=8.5)
ax.set_ylabel('% votos nulos y blancos', fontsize=10)
ax.set_title('Evolución del voto nulo y blanco en elecciones presidenciales chilenas\n1989–2025',
             fontsize=12, fontweight='bold', pad=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# Leyenda de regímenes
from matplotlib.lines import Line2D
leyenda = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4C72B0', markersize=9,
           label='Inscripción voluntaria, voto obligatorio'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#DD8452', markersize=9,
           label='Voto voluntario'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#C44E52', markersize=9,
           label='Obligatorio universal (inscripción automática)'),
]
ax.legend(handles=leyenda, fontsize=8.5, loc='upper left')

plt.tight_layout()
plt.show()

---
## 5. Tabla dinámica: Promedio de voto nulo/blanco por régimen de votación

Agrupación de los resultados por tipo de régimen para comparar el comportamiento promedio del voto nulo y blanco según el sistema electoral vigente.

In [ ]:
pivot_regimen = df.pivot_table(
    values='% votos nulos y blancos',
    index='Régimen del voto',
    aggfunc=['mean', 'min', 'max', 'count']
).round(4)

pivot_regimen.columns = ['Promedio', 'Mínimo', 'Máximo', 'N elecciones']
pivot_regimen['Promedio'] = (pivot_regimen['Promedio'] * 100).map('{:.2f}%'.format)
pivot_regimen['Mínimo']   = (pivot_regimen['Mínimo']   * 100).map('{:.2f}%'.format)
pivot_regimen['Máximo']   = (pivot_regimen['Máximo']   * 100).map('{:.2f}%'.format)

pivot_regimen

---
## 6. Tabla dinámica: Votos totales y nulos/blancos por elección

Vista comparativa del volumen absoluto de votos para cada proceso electoral.

In [ ]:
pivot_votos = df[[
    'Año Elecciones', 'candidato 1', 'candidato 2',
    'Votos totales', 'votos nulos', 'Votos blancos',
    'Suma votos nulos y blancos', '% votos nulos y blancos'
]].copy()

pivot_votos['% votos nulos y blancos'] = (
    pivot_votos['% votos nulos y blancos'] * 100
).map('{:.2f}%'.format)

for col in ['Votos totales', 'votos nulos', 'Votos blancos', 'Suma votos nulos y blancos']:
    pivot_votos[col] = pivot_votos[col].map('{:,}'.format)

pivot_votos.set_index('Año Elecciones')

---

## Notas finales

- **Fuente de los datos:** Servicio Electoral de Chile (SERVEL) y Biblioteca del Congreso Nacional (BCN).
- **Elaboración:** Base construida manualmente. Ver `README.md` para la documentación completa del proceso de limpieza y las decisiones editoriales.
- **Observación sobre 2025:** el incremento del porcentaje de voto nulo y blanco en 2025 (7,06%) está directamente asociado al cambio de régimen de votación hacia la obligatoriedad universal con inscripción automática, que incorporó al padrón a millones de electores previamente no participantes.
- **Próximo paso:** asignar un valor de posición ideológica a cada candidato (escala izquierda–derecha) para analizar si la distancia ideológica entre candidatos se correlaciona con el nivel de voto nulo y blanco.